# Exploração do Banco de Dados — RadioSkySim

Este notebook conecta ao **PostgreSQL** do RadioSkySim (configuração em
`settings.toml`, seção `[DB]`) e explora as tabelas persistidas pelos pacotes:

- `jobs` — execuções (foreground/background)
- `simulations` — registros de simulações de TOD (`SimOrchestrator`)
- `mapmaking_results` — resultados de mapmaking (TOD → mapas)
- `radiosources` — catálogo de fontes de rádio
- `satid`, `orbital_elements`, `signal`, `sat_signal_link` — satélites e sinais
- `solar_flux_cache` — cache de fluxo solar
- `beamparamsdb` — parâmetros de beam
- tabelas de skymap (HEALPix)

Tudo é feito com a engine **síncrona** (`psycopg2`), exposta por
`radioskysims_core.db.engine.Database`.


In [1]:
# ── Âncora de diretório: raiz do projeto (CWD-independente) ──
import os

from radioskysims_core.paths import get_project_root

os.chdir(get_project_root())
print("cwd →", os.getcwd())

import pandas as pd
from sqlalchemy import inspect

from radioskysims_core.db.engine import Database


cwd → /share/lbarosi/RadioSkySimSuite


In [2]:
# ── Conexão (lê [DB] de settings.toml) ──
db = Database()
print(db)  # URL sanitizada + tabelas visíveis pelo inspector (sem registrar modelos)

print("Teste de conexão:", db.test_connection())


<Database Engine — postgresql+psycopg2>
───────────────────────────────────────
URL: localhost:5432/radioskysims
Tabelas: beamparamsdb, radiosources, simulations, skymaprecord, solar_flux_cache, sat_signal_link, signal, satid, orbital_elements, sensor_readings

Teste de conexão: True


In [3]:
# ── Registra TODOS os modelos SQLModel (schema plugável) ──
# Sem estes imports o SQLModel.metadata fica vazio e o inspector não enxerga
# as tabelas da aplicação (mesmo padrão de src/radioskysims/infrastructure/db/init_db.py).
from radioskysims_core.db import jobs  # noqa: F401
from radioskysims_skymap.infrastructure.db import skymap_models  # noqa: F401
from radioskysims_sources.infrastructure.db import solar_flux_db_model  # noqa: F401
from radioskysims_sources.infrastructure.db.satellite import models as satellite_models  # noqa: F401
from radioskysims_sources.infrastructure.db import radiosources_db_model  # noqa: F401
from radioskysims_simulation.infrastructure.db import simulation_db_model  # noqa: F401
from radioskysims_simulation.infrastructure.db import mapmaking_db_model  # noqa: F401
from radioskysims_beam.infrastructure.db import beam_db_model  # noqa: F401

inspector = inspect(db.engine)
tables = sorted(inspector.get_table_names())
print(f"{len(tables)} tabela(s) no schema atual:")
for t in tables:
    print(" •", t)


10 tabela(s) no schema atual:
 • beamparamsdb
 • orbital_elements
 • radiosources
 • sat_signal_link
 • satid
 • sensor_readings
 • signal
 • simulations
 • skymaprecord
 • solar_flux_cache


In [4]:
# ── Contagem de linhas por tabela ──
rows = []
for t in tables:
    try:
        n = pd.read_sql_query(f'SELECT count(*) AS n FROM "{t}"', db.engine).iloc[0, 0]
        rows.append({"tabela": t, "linhas": int(n)})
    except Exception as exc:
        rows.append({"tabela": t, "linhas": f"erro: {type(exc).__name__}"})
        print(f"⚠️ {t}: {type(exc).__name__}: {exc}")

counts = pd.DataFrame(rows).sort_values("linhas", ascending=False)
counts


,tabela,linhas
1,orbital_elements,8709
3,sat_signal_link,726
2,radiosources,323
4,satid,170
6,signal,19
7,simulations,17
9,solar_flux_cache,7
0,beamparamsdb,0
5,sensor_readings,0
8,skymaprecord,0


In [5]:
# ── Colunas de cada tabela ──
for t in tables:
    cols = inspector.get_columns(t)
    print(f"\n{t} ({len(cols)} colunas):")
    print(", ".join(c["name"] for c in cols))



beamparamsdb (9 colunas):
id, polarizations, frequencies, has_phase, fwhm, directivity, main_lobe_directivity, directivity_db, main_lobe_efficiency

orbital_elements (41 colunas):
id, ccsds_omm_vers, comment, creation_date, originator, object_name, object_id, norad_id, center_name, ref_frame, time_system, mean_element_theory, epoch, mean_motion, eccentricity, inclination, ra_of_asc_node, arg_of_pericenter, mean_anomaly, ephemeris_type, classification_type, element_set_no, rev_at_epoch, bstar, mean_motion_dot, mean_motion_ddot, semimajor_axis, period, apoapsis, periapsis, object_type, rcs_size, country_code, launch_date, site, decay_date, file, gp_id, tle_line0, tle_line1, tle_line2

radiosources (6 colunas):
id, ra_deg, dec_deg, nu0_mhz, s0_jy, alpha

sat_signal_link (2 colunas):
sat_id, signal_id

satid (11 colunas):
norad_id, name, period, inclination, apogee, perigee, eccentricity, gp_age, constellation, type, prn

sensor_readings (3 colunas):
id, received_at, payload

signal (4 co

In [6]:
# ── Amostra (top 5) de cada tabela ──
for t in tables:
    try:
        df = pd.read_sql_query(f'SELECT * FROM "{t}" LIMIT 5', db.engine)
        print(f"\n=== {t} — {len(df)} linha(s) ===")
        display(df)
    except Exception as exc:
        print(f"⚠️ {t}: {type(exc).__name__}: {exc}")



=== beamparamsdb — 0 linha(s) ===


,id,polarizations,frequencies,has_phase,fwhm,directivity,main_lobe_directivity,directivity_db,main_lobe_efficiency



=== orbital_elements — 5 linha(s) ===


,id,ccsds_omm_vers,comment,creation_date,originator,object_name,object_id,norad_id,center_name,ref_frame,...,rcs_size,country_code,launch_date,site,decay_date,file,gp_id,tle_line0,tle_line1,tle_line2
0,1,3.0,GENERATED VIA SPACE-TRACK.ORG API,2025-05-15 04:48:17,18 SPCS,COSMOS 2492 (GLONASS),2014-012A,39620,EARTH,TEME,...,LARGE,CIS,2014-03-23,PKMTR,None,4728901,287611655,0 COSMOS 2492 (GLONASS),1 39620U 14012A 25133.99831591 -.00000041 0...,2 39620 65.5223 327.4729 0011618 330.1915 29...
1,2,3.0,GENERATED VIA SPACE-TRACK.ORG API,2025-05-14 06:56:18,18 SPCS,COSMOS 2492 (GLONASS),2014-012A,39620,EARTH,TEME,...,LARGE,CIS,2014-03-23,PKMTR,None,4728076,287520057,0 COSMOS 2492 (GLONASS),1 39620U 14012A 25133.99831588 -.00000041 0...,2 39620 65.5223 327.4729 0011619 330.1922 29...
2,3,3.0,GENERATED VIA SPACE-TRACK.ORG API,2025-05-14 03:37:07,18 SPCS,NAVSTAR 56 (USA 180),2004-045A,28474,EARTH,TEME,...,LARGE,US,2004-11-06,AFETR,None,4727914,287492415,0 NAVSTAR 56 (USA 180),1 28474U 04045A 25133.98191784 -.00000076 0...,2 28474 55.3351 340.9509 0166036 302.9833 87...
3,4,3.0,GENERATED VIA SPACE-TRACK.ORG API,2025-05-14 03:37:07,18 SPCS,NAVSTAR 51 (USA 166),2003-005A,27663,EARTH,TEME,...,LARGE,US,2003-01-29,AFETR,None,4727914,287492731,0 NAVSTAR 51 (USA 166),1 27663U 03005A 25133.97200928 .00000017 0...,2 27663 54.9829 230.8354 0142779 50.1412 114...
4,5,3.0,GENERATED VIA SPACE-TRACK.ORG API,2025-05-15 04:48:17,18 SPCS,GALILEO-PFM,2011-060A,37846,EARTH,TEME,...,LARGE,ESA,2011-10-21,FRGUI,None,4728901,287610139,0 GALILEO-PFM,1 37846U 11060A 25133.97064511 -.00000075 0...,2 37846 57.0981 352.5819 0003772 34.5662 325...



=== radiosources — 5 linha(s) ===


,id,ra_deg,dec_deg,nu0_mhz,s0_jy,alpha
0,120022+024737,180.094208,2.793833,1400.0,0.1002,1.232789
1,120023-010559,180.098458,-1.099806,1400.0,0.1539,0.826001
2,120045+040636,180.188750,4.110056,1400.0,0.3521,0.222758
3,120123+002829,180.346875,0.474833,1400.0,0.3784,0.391386
4,120205+024851,180.523250,2.814167,1400.0,0.2624,0.527034


... (10 outputs left out)


=== skymaprecord — 0 linha(s) ===


,id,nside,npix,unit,frame,field,freq_ghz,data_blob,created_at



=== solar_flux_cache — 5 linha(s) ===


,id,filename,start_utc,end_utc,source_url,created_at
0,13,solar_flux_20261010T155959_20261012T185959.nc,2026-10-10 15:59:59.999986,2026-10-12 18:59:59.999986,https://solar.nro.nao.ac.jp/norp/xdr/2026/10/n...,2026-10-13 20:09:08.552798
1,14,solar_flux_20261010T023600_20261012T033600.nc,2026-10-10 02:36:00.000000,2026-10-12 03:36:00.000000,https://solar.nro.nao.ac.jp/norp/xdr/2026/10/n...,2026-10-13 20:16:21.142138
2,15,solar_flux_20261010T024200_20261012T034200.nc,2026-10-10 02:42:00.000000,2026-10-12 03:42:00.000000,https://solar.nro.nao.ac.jp/norp/xdr/2026/10/n...,2026-10-13 20:30:50.792298
3,16,solar_flux_20261010T025400_20261012T035400.nc,2026-10-10 02:54:00.000000,2026-10-12 03:54:00.000000,https://solar.nro.nao.ac.jp/norp/xdr/2026/10/n...,2026-10-13 20:45:46.733342
4,17,solar_flux_20261010T030000_20261013T030000.nc,2026-10-10 03:00:00.000000,2026-10-13 03:00:00.000000,https://solar.nro.nao.ac.jp/norp/xdr/2026/10/n...,2026-10-13 21:00:41.747733


In [ ]:
# ── Visões específicas ──
# Jobs mais recentes (status/duração)
if "jobs" in tables:
    jobs_df = pd.read_sql_query(
        "SELECT id, name, status, progress_percent, progress_step, "
        'created_at, updated_at, error FROM "jobs" '
        "ORDER BY created_at DESC LIMIT 10",
        db.engine,
    )
    print(f"Últimos {len(jobs_df)} jobs:")
    display(jobs_df)


In [8]:
# Simulações registradas (resumo escalar)
if "simulations" in tables:
    sims_df = pd.read_sql_query(
        "SELECT id, telescope_name, freq_min_mhz, freq_max_mhz, n_channels, "
        "n_satellites, n_radio_sources, time_start_utc, time_end_utc, "
        "computing_time_s, dataset_size_mb, saved, registered, created_at "
        'FROM "simulations" ORDER BY created_at DESC LIMIT 10',
        db.engine,
    )
    print(f"Últimas {len(sims_df)} simulações:")
    display(sims_df)


Últimas 10 simulações:


,id,telescope_name,freq_min_mhz,freq_max_mhz,n_channels,n_satellites,n_radio_sources,time_start_utc,time_end_utc,computing_time_s,dataset_size_mb,saved,registered,created_at
0,17,uirapuru,1130.0,1130.0,1,146.0,None,2026-08-31 00:00:00,2026-09-07 00:00:00,10.337536,NaN,True,True,2026-08-26 21:24:32.833692
1,16,uirapuru,1130.0,1130.0,1,146.0,None,2026-08-31 00:00:00,2026-09-07 00:00:00,10.138472,NaN,True,True,2026-08-26 21:16:48.746017
2,15,uirapuru,1130.0,1130.0,1,146.0,None,2026-08-31 00:00:00,2026-09-07 00:00:00,10.267369,NaN,True,True,2026-08-26 21:14:57.784318
3,14,uirapuru,1130.0,1130.0,1,146.0,None,2026-08-31 00:00:00,2026-09-07 00:00:00,12.096499,NaN,True,True,2026-08-26 21:06:46.892426
4,13,uirapuru,1000.0,1250.0,5000,NaN,None,2025-02-27 12:00:00,2025-02-28 12:00:00,15.757919,7482.809166,True,True,2026-08-25 18:54:07.562367
5,12,uirapuru,1000.0,1250.0,5000,146.0,None,2025-02-27 00:00:00,2025-02-28 00:00:00,42.250494,NaN,True,True,2026-08-18 21:12:31.717678
6,11,uirapuru,1000.0,1250.0,5000,146.0,None,2025-05-09 00:00:00,2025-05-10 00:00:00,35.112657,NaN,True,True,2026-08-17 13:13:49.311288
7,10,uirapuru,1000.0,1250.0,5000,146.0,None,2025-05-09 00:00:00,2025-05-10 00:00:00,13.672619,NaN,True,True,2026-08-16 21:38:54.968620
8,9,uirapuru,1000.0,1250.0,5000,146.0,None,2025-05-09 00:00:00,2025-05-10 00:00:00,24.419308,NaN,True,True,2026-08-16 21:36:03.306670
9,8,uirapuru,1000.0,1250.0,5000,146.0,None,2025-05-09 00:00:00,2025-05-10 00:00:00,17.299955,NaN,True,True,2026-08-15 19:26:16.484622


In [9]:
# Satélites persistidos (TLE) — amostra
if "satid" in tables:
    sats_df = pd.read_sql_query('SELECT * FROM "satid" LIMIT 10', db.engine)
    print(f"Amostra de satélites ({len(sats_df)}):")
    display(sats_df)


Amostra de satélites (10):


,norad_id,name,period,inclination,apogee,perigee,eccentricity,gp_age,constellation,type,prn
0,37846,GSAT0101 (GALILEO-PFM),844.70,56.97,23227.0,23217.0,0.000166,1.81,GALILEO,GNSS,None
1,37847,GSAT0102 (GALILEO-FM2),844.69,56.97,23227.0,23217.0,0.000171,0.70,GALILEO,GNSS,None
2,38857,GSAT0103 (GALILEO-FM3),844.71,55.84,23230.0,23215.0,0.000262,1.18,GALILEO,GNSS,None
3,39199,IRNSS-1A,1436.01,35.84,35866.0,35703.0,0.001934,3.39,IRNSS,GNSS,None
4,39635,IRNSS-1B,1436.06,28.83,35874.0,35698.0,0.002092,1.03,IRNSS,GNSS,None
5,40269,IRNSS-1C,1436.04,6.66,35874.0,35697.0,0.002103,0.34,IRNSS,GNSS,None
6,40315,COSMOS 2501 (702K),675.74,63.62,19187.0,19072.0,0.002254,2.53,GLONASS,GNSS,None
7,41241,IRNSS-1E,1436.03,33.36,35873.0,35698.0,0.002076,0.51,IRNSS,GNSS,None
8,41469,IRNSS-1G,1436.10,5.34,35809.0,35765.0,0.000520,0.36,IRNSS,GNSS,None
9,43286,IRNSS-1I,1436.07,28.89,35866.0,35706.0,0.001889,0.49,IRNSS,GNSS,None


## Conclusão

Acima temos um panorama do conteúdo persistido no banco. Dependendo do estado
do schema (tabelas criadas via `python -m radioskysims.infrastructure.db.init_db`)
algumas tabelas podem estar vazias — é esperado em um ambiente recém-inicializado.
